In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
#1

def get_json(BASE_URL: str) -> dict:
    response = requests.get(BASE_URL)
    if response.status_code == 200: 
        return response.json()
    else: 
        raise Exception(f'Error, status_code {response.status_code}')

get_json(BASE_URL + 'status.json')['service']
#get_json(BASE_URL + 'orders/page-99.json')


'Northern Star Shop API'

In [0]:
#2
os.path.join(RAW_DIR,'orders/page-1.json')
def fetch_all(first_url: str) -> list:
    response = get_json(first_url)
    data = response['data']
    next_page = response['next']
    while next_page:
        response = get_json(urljoin(first_url, next_page))
        data += response['data']
        next_page = response['next']
    return data

fetch_all(BASE_URL + 'orders/page-1.json')


[{'order_id': 'NSS-00001',
  'order_date': '2026-08-15',
  'customer': {'name': 'NIKOLA STOJANOVIĆ', 'city': 'Zrenjanin'},
  'product_id': 1002,
  'quantity': 1,
  'unit_price': 6990.0,
  'payment': 'cash'},
 {'order_id': ' nss-00002 ',
  'order_date': '2026-08-15',
  'customer': {'name': ' Jovana Pavlović ', 'city': 'Čačak'},
  'product_id': 1094,
  'quantity': 3,
  'unit_price': 1290.0,
  'payment': 'card'},
 {'order_id': 'NSS-00003',
  'order_date': '2026-08-15',
  'customer': {'name': 'Sara Živković', 'city': 'Beograd'},
  'product_id': 1041,
  'quantity': 1,
  'unit_price': 6990.0,
  'payment': 'card'},
 {'order_id': 'NSS-00004',
  'order_date': '2026-08-15',
  'customer': {'name': 'Sara Đorđević', 'city': ' Subotica '},
  'product_id': 1075,
  'quantity': 1,
  'unit_price': 990.0,
  'payment': 'cash'},
 {'order_id': 'NSS-00005',
  'order_date': '2026-08-15',
  'customer': {'name': 'Stefan Popović', 'city': 'Niš'},
  'product_id': 1044,
  'quantity': 1,
  'unit_price': 1290.0,
  '

In [0]:
#3

def save_json(records: list, path: str) ->None:
    with open(path, 'w') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    return None

save_json([{'id': 1, 'name': 'Jovan'}], os.path.join(RAW_DIR, 'orders.json'))

In [0]:
#4

def download_file(url: str, path: str) -> int:
    response = requests.get(url, timeout=30)
    response.raise_for_status() 
    with open(path,'wb') as f:
        f.write(response.content)
    return len(response.content)

#download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))
#download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR,'products.csv'))

In [0]:
def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(raw_dir, 'orders.json'))
    teams = download_file(FILES_URL + 'teams.csv', os.path.join(raw_dir, 'teams.csv'))
    products = download_file(FILES_URL + 'products.csv', os.path.join(raw_dir, 'products.csv'))
    matches = download_file(FILES_URL + 'matches.parquet', os.path.join(raw_dir, 'matches.parquet'))
    orders = pd.json_normalize(orders, sep='_')
    orders = orders.rename(columns= {'customer_name': 'customer', 'customer_city': 'city'})
    teams = pd.read_csv(os.path.join(raw_dir, 'teams.csv'))
    products = pd.read_csv(os.path.join(raw_dir, 'products.csv'))
    matches = pd.read_parquet(os.path.join(raw_dir, 'matches.parquet'))
    return {
        'oreders': orders,
        'teams': teams,
        'products': products,
        'matches': matches
    }
extract(RAW_DIR)

{'oreders':          order_id  order_date  ...           customer        city
 0       NSS-00001  2026-08-15  ...  NIKOLA STOJANOVIĆ   Zrenjanin
 1      nss-00002   2026-08-15  ...   Jovana Pavlović        Čačak
 2       NSS-00003  2026-08-15  ...      Sara Živković     Beograd
 3       NSS-00004  2026-08-15  ...      Sara Đorđević   Subotica 
 4       NSS-00005  2026-08-15  ...     Stefan Popović         Niš
 ...           ...         ...  ...                ...         ...
 1847    NSS-01836  2026-11-15  ...     MILICA POPOVIĆ   SUBOTICA 
 1848    NSS-01837  2026-11-15  ...    marko kovačević       Čačak
 1849    NSS-01838  2026-11-15  ...      Marko Nikolić      Čačak 
 1850    NSS-01839  2026-11-15  ...    Jelena Živković   Zrenjanin
 1851    NSS-01840  2026-11-15  ...     ĐORĐE ĐORĐEVIĆ    Novi Sad
 
 [1852 rows x 8 columns],
 'teams':     team_id code             team_name          city  founded_year
 0         1  NPM    Northport Mariners     Northport          1894
 1         2

In [0]:

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates(subset = 'product_id')
    df['team_code'] = df['team_code'].str.strip().str.upper()
    df['category'] = df['category'].str.strip().str.lower()
    df['price'] =  pd.to_numeric(df['price'].str.replace(' RSD', '', regex=False).str.replace(',', '.'))
    df['active'] = df['active'].str.upper().isin(['TRUE','YES'])
    return df

clean_products(pd.read_csv(os.path.join(RAW_DIR, 'products.csv')))


,product_id,sku,product_name,team_code,category,price,active
0,1001,NPM-JER-1001,NPM Home Jersey,NPM,jerseys,7490.0,True
1,1002,NPM-JER-1002,NPM Away Jersey,NPM,jerseys,6990.0,True
2,1003,NPM-SCA-1003,NPM Scarf,NPM,scarves,1490.0,True
3,1004,NPM-CAP-1004,NPM Cap,NPM,caps,2290.0,True
4,1005,NPM-ACC-1005,NPM Mug,NPM,accessories,990.0,True
...,...,...,...,...,...,...,...
100,1101,NSL-BAL-1101,NSL Match Ball,NaN,balls,7490.0,True
101,1102,NSL-BAL-1102,NSL Training Ball,NaN,balls,2990.0,True
102,1103,NSL-ACC-1103,NSL Water Bottle,NaN,accessories,890.0,True
103,1104,NSL-ACC-1104,NSL Backpack,NaN,accessories,3990.0,True
